# Function 2: 2D black-box maximisation

Imagine a black box, or a mystery ML model, that takes two numbers as input and returns a log-likelihood score. Your goal is to maximise that score, but each output is noisy, and depending on where you start, you might get stuck in a local optimum. To tackle this, you use Bayesian optimisation, which selects the next inputs based on what it has learned so far. It balances exploration with exploitation, making it well suited to noisy outputs and complex functions with many local peaks.

## How this notebook works

1. **Data loading:** observations are read from `data.csv` and the raw noise-variance floor is converted to the normalised alpha scale used by scikit-learn.
2. **Final GP:** a constant × Matérn 3/2 ARD Gaussian process with the noise floor as alpha, checked with leave-one-out cross-validation.
3. **Global search:** posterior mean, EI, PI and UCB on a dense grid over the unit square.
4. **Local refinement:** a search of ±one fitted x1 length scale around the incumbent, holding x2 fixed.
5. **Noise sensitivity:** the GP is refitted at different assumed noise levels, and EI and PI are recomputed against a latent incumbent (the highest GP mean at an observed point).
6. **Candidates and final-round decision:** the local candidates are printed in submission format. For the final round the EI optimum of the local x1 search (x1 = 0.494947, x2 = 0.55) was chosen after R12 (x1 = 0.515) scored lower than the incumbent; the closing cell records the reasoning.

Everything is derived from `data.csv`: append new rows and re-run, with no code edits. `QUICK_MODE = True` uses fewer optimiser restarts and coarser grids; set it to `False` for the fuller run.

**Data-handling choices.**

- *Noise.* Function 2 is noisy. The noise floor (raw variance 0.013264, SD about 0.115) was estimated from repeated evaluations and is held fixed, not fitted.
- *Input format.* Candidates are printed to 6 decimals, as the portal requires.

In [1]:
# =============================================================================
# IMPORTS
# =============================================================================

import warnings

import numpy as np
import pandas as pd

from scipy.stats import norm
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel as C, Matern
from sklearn.model_selection import LeaveOneOut, cross_val_predict

warnings.filterwarnings("ignore", category=UserWarning)

In [2]:
# =============================================================================
# SETTINGS
# =============================================================================
# QUICK_MODE=True reduces optimiser restarts and grid sizes for fast iteration.
# Set QUICK_MODE=False for the fuller weekly run.

QUICK_MODE = True

DATA_PATH = "../data/data.csv"
FUNCTION_ID = 2
RANDOM_SEED = 42

# Empirical RAW y-variance floor estimated from repeated evaluations of the
# same locations (pooled within-location SD ~= 0.11517, variance ~= 0.013264).
# Because normalize_y=True, it is converted below into the normalised alpha
# scale actually used by GaussianProcessRegressor.
RAW_NOISE_VARIANCE_FLOOR = 0.013264
NOISE_SENSITIVITY = [1.0, 0.5]    # noise-floor multipliers for the sensitivity check

# GP kernel: constant * Matérn-3/2 with one length scale per input (ARD)
CONSTANT_BOUNDS = (1e-3, 1e3)
LENGTH_SCALE_BOUNDS = (1e-2, 1e2)
MATERN_NU = 1.5
NORMALIZE_Y = True

# Optimiser restarts
N_RESTARTS_DIAGNOSTIC = 3 if QUICK_MODE else 10     # leave-one-out check
N_RESTARTS_FINAL = 8 if QUICK_MODE else 25          # final fits

# Acquisition functions
EI_XI = 0.01
PI_XI = 0.01
UCB_KAPPA = 2.0

# Search resolution
GLOBAL_GRID_SIZE = 250 if QUICK_MODE else 1000      # points per axis over [0, 1]^2
LOCAL_GRID_SIZE = 1000 if QUICK_MODE else 5000      # points along the local x1 line

The choices were deliberately conservative:

* **Quick mode = True:** reduces GP optimiser restarts and grid resolution. Set `QUICK_MODE = False` for the fuller run.
* **Raw noise variance floor = 0.013264:** derived from repeated observations (observation-noise SD of about 0.115). It is converted to the normalised alpha scale used by scikit-learn.
* **EI ξ = 0.01:** a low exploration parameter, so EI is primarily exploitative.
* **UCB κ = 2.0:** a moderate exploration/exploitation balance, useful as a comparator to EI.
* **250 × 250 grid:** 62,500 candidates in the 2D [0, 1]² space.

In [3]:
# =============================================================================
# DATA LOADING
# =============================================================================
# Loads the centrally prepared BBO dataset and selects the active function.
# Active input dimensions are inferred from the x-columns that contain data.

data = pd.read_csv(DATA_PATH, float_precision="round_trip")
function_data = data[data["function"] == FUNCTION_ID].copy()

feature_cols = [
    f"x{i}" for i in range(1, 9)
    if function_data[f"x{i}"].notna().any()
]

# Modelling arrays
X = function_data[feature_cols].to_numpy()
y = function_data["y"].to_numpy()

# Observation table
df = function_data[[*feature_cols, "y", "round"]].copy()
df["Stage"] = df["round"].apply(
    lambda r: "Initial Data" if r == 0 else f"Week {r}"
)
df = df.rename(columns={"round": "Round"})
df = df[[*feature_cols, "y", "Stage", "Round"]]

# Convert the raw-y noise floor to sklearn's normalised-y alpha scale.
y_variance_population = float(np.var(y, ddof=0))
ALPHA_FLOOR = (
    RAW_NOISE_VARIANCE_FLOOR / y_variance_population
    if NORMALIZE_Y
    else RAW_NOISE_VARIANCE_FLOOR
)

print(f"Function {FUNCTION_ID} observations: {len(df)}")
print(f"Active features: {feature_cols}")
print(f"Rounds: R0–R{df['Round'].max()}")
print(f"Observed y variance:      {y_variance_population:.6f}")
print(f"Normalized alpha floor:  {ALPHA_FLOOR:.6f}")

Function 2 observations: 22
Active features: ['x1', 'x2']
Rounds: R0–R12
Observed y variance:      0.071934
Normalized alpha floor:  0.184390


In [4]:
# =============================================================================
# MODEL HELPERS
# =============================================================================


def make_gp(alpha, n_restarts):
    """Fresh GP: constant * Matérn-3/2 ARD kernel with fixed noise alpha."""
    kernel = C(1.0, CONSTANT_BOUNDS) * Matern(
        length_scale=np.ones(X.shape[1]),
        length_scale_bounds=LENGTH_SCALE_BOUNDS,
        nu=MATERN_NU,
    )
    return GaussianProcessRegressor(
        kernel=kernel,
        n_restarts_optimizer=n_restarts,
        alpha=float(alpha),
        normalize_y=NORMALIZE_Y,
        random_state=RANDOM_SEED,
    )


def acquisition_values(mu, sigma, y_best):
    """EI, PI and UCB for posterior mean/std arrays, relative to y_best."""
    sigma_safe = np.maximum(sigma, 1e-12)
    z_ei = (mu - y_best - EI_XI) / sigma_safe
    ei = (mu - y_best - EI_XI) * norm.cdf(z_ei) + sigma_safe * norm.pdf(z_ei)
    pi = norm.cdf((mu - y_best - PI_XI) / sigma_safe)
    ucb = mu + UCB_KAPPA * sigma
    return ei, pi, ucb


def acquisition_winners(gp, candidates, y_best):
    """Best candidate under posterior mean, EI, PI and UCB, as a DataFrame."""
    mu, sigma = gp.predict(candidates, return_std=True)
    ei, pi, ucb = acquisition_values(mu, sigma, y_best)
    rows = []
    for method, score in [("Posterior Mean", mu), ("EI", ei), ("PI", pi), ("UCB", ucb)]:
        i = int(np.argmax(score))
        rows.append({"Method": method, "x1": candidates[i, 0], "x2": candidates[i, 1],
                     "Pred_y": mu[i], "Pred_Std": sigma[i], "EI": ei[i], "PI": pi[i], "UCB": ucb[i]})
    return pd.DataFrame(rows)


def format_point(x):
    """Format a point as the dash-separated six-decimal submission string."""
    return "-".join(f"{v:.6f}" for v in x)

In [5]:
# =============================================================================
# FINAL GP MODEL
# =============================================================================
# Matérn-3/2 ARD GP with the empirical noise floor as alpha. Leave-one-out
# cross-validation (fewer restarts) gives a quick check of predictive accuracy.

loo_pred = cross_val_predict(make_gp(ALPHA_FLOOR, N_RESTARTS_DIAGNOSTIC), X, y, cv=LeaveOneOut())
print(f"GP LOOCV: RMSE={np.sqrt(np.mean((y - loo_pred) ** 2)):.6f}, "
      f"MAE={np.mean(np.abs(y - loo_pred)):.6f}")

gp_ard = make_gp(ALPHA_FLOOR, N_RESTARTS_FINAL).fit(X, y)
length_scales = np.atleast_1d(gp_ard.kernel_.k2.length_scale)

print(f"Kernel: {gp_ard.kernel_}")
print(f"Alpha / noise variance: {ALPHA_FLOOR:.6f}")
print(f"Log marginal likelihood: {gp_ard.log_marginal_likelihood_value_:.6f}")
print("ARD length scales: " + ", ".join(f"{f}={ls:.6f}" for f, ls in zip(feature_cols, length_scales)))

inc_idx = int(np.argmax(y))
x_inc, y_best = X[inc_idx], float(y[inc_idx])
print(f"Incumbent: R{int(df['Round'].iloc[inc_idx])}, x1={x_inc[0]:.6f}, x2={x_inc[1]:.6f}, y={y_best:.8f}")

GP LOOCV: RMSE=0.190137, MAE=0.121426
Kernel: 1.04**2 * Matern(length_scale=[0.0526, 100], nu=1.5)
Alpha / noise variance: 0.184390
Log marginal likelihood: -21.448679
ARD length scales: x1=0.052581, x2=100.000000
Incumbent: R11, x1=0.507000, x2=0.550000, y=0.74903986


In [6]:
# =============================================================================
# CANDIDATE SEARCH
# =============================================================================

# ---- 1. Global search: dense grid over [0, 1]^2 ---------------------------------
x_axis = np.linspace(0.0, 1.0, GLOBAL_GRID_SIZE)
X_grid = np.column_stack([a.ravel() for a in np.meshgrid(x_axis, x_axis)])

print(f"GLOBAL ACQUISITION COMPARISON | {len(X_grid):,} grid points | incumbent y = {y_best:.8f}")
print(acquisition_winners(gp_ard, X_grid, y_best).to_string(index=False))

# ---- 2. Local refinement: +/- one fitted x1 length scale around the incumbent ----
# x2 is held at the incumbent value.
ls_x1 = float(length_scales[0])
x1_low, x1_high = max(0.0, x_inc[0] - ls_x1), min(1.0, x_inc[0] + ls_x1)
x1_line = np.linspace(x1_low, x1_high, LOCAL_GRID_SIZE)
X_local = np.column_stack([x1_line, np.full_like(x1_line, x_inc[1])])

local_results = acquisition_winners(gp_ard, X_local, y_best)
local_results["Distance_x1_from_incumbent"] = (local_results["x1"] - x_inc[0]).abs()

print(f"\nLOCAL x1 REFINEMENT | x1 length scale {ls_x1:.6f} | x1 range [{x1_low:.6f}, {x1_high:.6f}] | x2 fixed at {x_inc[1]:.6f}")
print(local_results.to_string(index=False))

# ---- 3. Noise sensitivity ---------------------------------------------------------
# The best raw observation may include a positive noise shock, so EI/PI are
# re-run against a LATENT incumbent (highest GP mean at an observed location)
# under each assumed noise level. This is a sensitivity check only.
noise_rows = []
for multiplier in NOISE_SENSITIVITY:
    gp_noise = make_gp(multiplier * ALPHA_FLOOR, N_RESTARTS_FINAL).fit(X, y)
    mu_obs = gp_noise.predict(X)
    latent = float(mu_obs.max())
    winners = acquisition_winners(gp_noise, X_grid, latent)
    winners.insert(0, "Noise_Floor", f"{multiplier}x")
    winners["Latent_Incumbent"] = latent
    noise_rows.append(winners)

print("\nNOISE SENSITIVITY (global grid, latent incumbent)")
print(pd.concat(noise_rows).to_string(index=False))

# ---- 4. Local candidates in submission format -------------------------------------
print("\nLocal candidates (submission format):")
for _, row in local_results.iterrows():
    print(f"{row['Method']:<15} {format_point([row['x1'], row['x2']])}")

GLOBAL ACQUISITION COMPARISON | 62,500 grid points | incumbent y = 0.74903986
        Method       x1       x2   Pred_y  Pred_Std       EI       PI      UCB
Posterior Mean 0.506024 0.566265 0.650014  0.068472 0.001621 0.055663 0.786958
            EI 0.979920 0.000000 0.390628  0.277043 0.011835 0.091792 0.944713
            PI 0.971888 0.196787 0.392842  0.275608 0.011802 0.091975 0.944058
           UCB 0.983936 0.000000 0.389649  0.277576 0.011833 0.091632 0.944801

LOCAL x1 REFINEMENT | x1 length scale 0.052581 | x1 range [0.454419, 0.559581] | x2 fixed at 0.550000
        Method       x1   x2   Pred_y  Pred_Std       EI       PI      UCB  Distance_x1_from_incumbent
Posterior Mean 0.506316 0.55 0.650050  0.067889 0.001558 0.054202 0.785829                    0.000684
            EI 0.494947 0.55 0.610058  0.105981 0.003837 0.079899 0.822019                    0.012053
            PI 0.497789 0.55 0.626314  0.095869 0.003638 0.083112 0.818053                    0.009211
           U

Final input: 0.494947-0.550000

Decision Ratioanle

For the final round, the objective is to maximise expected improvement over the current best observed score of 0.749040.

GP model identifies a promising region around x1≈0.507, where the two highest scores were achieved. The most recent evaluation at x1=0.515 produced a lower score of 0.605345. This suggests that further exploration towards lower x1 values may be worthwhile (mindful of noise).

Decision was made to use EI optimum at x1=0.494947, while holding x2=0.550000 constant. This  is supported by the local acquisition analysis. It offers a balance between predicted performance and uncertainty.